# Example code showing the use of the API Laser Tracker

## Import API python module

In [ ]:
from api_lasertracker import ltpy as lt
print (f'Laser Tracker SDK module imported. Version: {lt.__version__}')

## Import other python modules

In [ ]:
from time import sleep
from IPython.display import clear_output, display
import math

## For any documentation help.

In [ ]:
help (lt)

## Create the instance for the device interface class

In [ ]:
dev = lt.LaserTrackerInterface()

## Connect the Laser Tracker and get the device information

In [ ]:
ret = dev.connect(ip_address='192.168.0.169', prm_files_path='~/Logging')
if ret != lt.ErrorType.Success:
    msg = dev.get_error_message(ret)
    print ('Connection Failed: {}'.format(msg))
else:
    info = dev.get_device_information()
    version = dev.get_sdk_version_number()
    print (f'Model: {info.device_model}\nSerial Number: {info.serial_number}\nDevice Firmware Version: {info.device_firmware_version}\nSDK Version: {version}')

## After connection, home the laser tracker

In [ ]:
ret = dev.home(lt.SmrSize.OneHalfInch)
if ret != lt.ErrorType.Success:
    msg = dev.get_error_message(ret)
    print (f'Homing Failed: {msg}')
else:
    print ('Homing succeeded')

## Jogging / Moving the laser tracker to a predefined position

### Using cartesian coordinates

In [ ]:
pt = lt.Vector3()
pt.x = 0.0
pt.y = 3000.0
pt.z = 5.0

ret = dev.point_to_cartesian(position = pt)
if ret != lt.ErrorType.Success:
    msg = dev.get_error_message(ret)
    print (f'Point to Failed: {msg}')
else:
    print ('Point to succeeded')

### Using spherical coordinates

In [ ]:
pt = lt.PolarVector3()
pt.az = -2.0
pt.el = 5.0
pt.distance = 3000.0

ret = dev.point_to_spherical(position = pt)
if ret != lt.ErrorType.Success:
    msg = dev.get_error_message(ret)
    print (f'Point to Failed: {msg}')
else:
    print ('Point to succeeded')

### Using absolute encoder angles

In [ ]:
az = 4.0
el = 10.0

ret = dev.jog_to(az = az, el = el)
if ret != lt.ErrorType.Success:
    msg = dev.get_error_message(ret)
    print (f'Jog to Failed: {msg}')
else:
    print ('Jog to succeeded')
# after jog to is completed, tracker will remain in position mode, so call switch_to_track()
# in position mode, it cannot track the SMR / any target
dev.switch_to_track()

### Using relative encoder angles

In [ ]:
az = -1.0
el = -1.0

ret = dev.jog_by(az = az, el = el)
if ret != lt.ErrorType.Success:
    msg = dev.get_error_message(ret)
    print (f'Jog by Failed: {msg}')
else:
    print ('Jog by succeeded')
# after jog by is completed, tracker will remain in position mode, so call switch_to_track()
# in position mode, it cannot track the SMR / any target
dev.switch_to_track()

## Searching

### Camera Based Search

The below function call will enable the camera in the background and whenever the tracker loses track, camera is activated and will start searching

In [ ]:
ret = dev.enable_camera_search(enable = True) # To disable, set it to False
if ret != lt.ErrorType.Success:
    msg = dev.get_error_message(ret)
    print (f'Camera Search Failed: {msg}')
else:
    print ('Camera Search enabled')

### Spiral Search

In [ ]:
ret = dev.spiral_search(estimated_distance = 3000.0, estimated_radius = 100.0, timeout = 40000)
if ret != lt.ErrorType.Success:
    msg = dev.get_error_message(ret)
    print (f'Spiral Search Failed: {msg}')
else:
    print ('Spiral Search succeeded')

## Opening iVision window to look at the video stream and to operate other iVision modes

In [ ]:
ret = dev.open_ivision_control_window()
if ret != lt.ErrorType.Success:
    msg = dev.get_error_message(ret)
    print (f'open ivision window Failed: {msg}')
else:
    print ('open ivision window succeeded')
    # in case there is no background UI thread, wait for the iVision window to be closed
    dev.wait_for_ivision_dialog_closing()
    print ('ivision window is closed')

## Getting real time data from the tracker and weather station data

In [ ]:
count = 0
while (count < 100):
    data = dev.get_realtime_data()
    data_str = f'Timestamp: {data[1].timestamp}\nX: {data[1].x:6.3f}\nY: {data[1].y:6.3f}\nZ: {data[1].z:6.3f}\nAZ: {data[1].az:6.3f}\nEL: {data[1].el:6.3f}\nDistance: {data[1].distance:6.3f}\nWarmedup Status: {data[1].is_warmed_up}\nMeasurement Valid Status: {data[1].is_measurement_valid}\nTracking Status: {data[1].is_tracking}\nOperation Mode: {data[1].operation_mode}\n'
    ws_data = dev.get_weather_station_data()
    data_str += f'Air Temperature: {ws_data[1].air_temperature:2.1f}\nAir Pressure: {ws_data[1].air_pressure:4.3f}\nAir Humidity: {ws_data[1].air_humidity}'
    clear_output(wait=True)
    print(data_str)
    sleep(0.1)
    count += 1

## Virtual Level

### Perform Virtual Level and Get the results

In [ ]:
vl_matrix = [1, 0, 0, 0, 1, 0, 0, 0, 1]
ret = dev.virtual_level()
if ret != lt.ErrorType.Success:
    msg = dev.get_error_message(ret)
    print (f'Virtual Level failed: {msg}')
else:
    # get virtual level matrix to save / store for future use
    # NOTE: After Virtual Level is performed, measurements are compensated with results if enabled, users should not compensate again.
    m = dev.get_virtual_level_matrix()
    if m[0] == lt.ErrorType.Success:
        print (f'VL Matrix: {m[1]}')
        vl_matrix = m[1]

### Set the virtual Level Matrix

In [ ]:
# Once the SDK is disconnected, VL matrix will no longer be in SDKs memory
# When user connects again, they can apply the existing matrix if saved from previous Virtual Level
# If device is moved from the current location, choose to do new virtual level instead of applying
ret = dev.set_virtual_level_matrix(matrix=vl_matrix)
if ret != lt.ErrorType.Success:
    msg = dev.get_error_message(ret)
    print (f'Set Virtual Level Matrix failed: {msg}')
else:
    print (f'VL Matrix set to SDK is: {vl_matrix}')

### Enable / Disable Virtual Level

In [ ]:
flag = True # False to disable
ret = dev.enable_virtual_level_frame(enable=flag)
if ret != lt.ErrorType.Success:
    msg = dev.get_error_message(ret)
    print (f'Enable/Disable Virtual Level failed: {msg}')
else:
    if flag:
        print (f'Virtual Level is enabled')
    else:
        print (f'Virtual Level is disabled')

## Target Measurements

### Static single point measurement

In [ ]:
ret = dev.get_single_point_measurement(averaging_time = 1000)
if ret[0] != lt.ErrorType.Success:
    msg = dev.get_error_message(ret)
    print (f'Get single point measurement failed: {msg}')
else:
    print (f'X: {ret[1].x:4.3f}, Y: {ret[1].y:4.3f}, Z: {ret[1].z:4.3f}')
    print (f'RMS Error: {ret[2]:4.3f}')

### Callback function for multi-smr measurements, temporal and spatial dynamic measurements, stable point data collection

In [ ]:
def measurement_callback_function(result : lt.MeasurementResult):
    if result.error_code == lt.ErrorType.Success:
        print (f'{result.measured_point.x:4.3f}, {result.measured_point.y:4.3f}, {result.measured_point.z:4.3f}')
    elif result.error_code == lt.ErrorType.MultiSmrMeasurementFinished:
        print (f'Multi SMR measurement is finished')
    else:
        msg = dev.get_error_message(ret)
        print (f'Error occurred during measurement: {msg}')

### iVision based multi-smr measurements

This feature is supported in Radian Pro and iLT trackers. Radian Plus/Core is not supported

In [ ]:
ret = dev.start_multi_smr_measurement(averaging_time = 1000, callback_function = measurement_callback_function)
if ret != lt.ErrorType.Success:
    msg = dev.get_error_message(ret)
    print (f'Start MSMR measurement Failed: {msg}')
else:
    print ('MSMR measurement started, wait for results')

## Continuous Automated Measurements

Let's assume we have some predefined targets which we want to measure repeatedly / or we know the rough location of the targets in Laser Tracker's frame of reference and we want to go there to measure the target

In [ ]:
targets = [lt.Vector3(275.460, 3691.276, 759.983), 
           lt.Vector3(167.936, 3607.068, 628.153)] # can add any number of targets to the list 

for target in targets:
    # Step 1: Move the tracker to predefined location
    ret = dev.point_to_cartesian(position = target)
    if ret != lt.ErrorType.Success:
        msg = dev.get_error_message(ret)
        print (f'Point to Failed: {msg}')
        break

    # Step 2: Check whether the tracker is locked on to SMR
    sleep(0.1) # time to safely flush any previous data due to internal queueing
    data = dev.get_realtime_data()
    if not data[1].is_tracking:
        # Turn on the camera search, allow it to lock on the target
        # Note that, spiral search can also be used here
        dev.enable_camera_search(enable = True)
        # wait for lock on
        is_tracker_locked_on = data[1].is_tracking
        while not is_tracker_locked_on:
            data = dev.get_realtime_data()
            is_tracker_locked_on = data[1].is_tracking
            sleep(0.1)

    # Step 3: At this point it is assumed that the tracker is locked on, wait for the measurement to be valid
    # Even if the tracker is locked on, it does not mean measurement is valid.
    measurement_valid = data[1].is_measurement_valid;
    while not measurement_valid:
        data = dev.get_realtime_data()
        measurement_valid = data[1].is_measurement_valid
        sleep(0.1)
    
    # Step 4: Collect the data
    ret = dev.get_single_point_measurement(averaging_time = 1000)
    if ret[0] != lt.ErrorType.Success:
        msg = dev.get_error_message(ret[0])
        print (f'Get single point measurement failed: {msg}')
    else:
        print (f'X: {ret[1].x:4.3f}, Y: {ret[1].y:4.3f}, Z: {ret[1].z:4.3f}')
        print (f'RMS Error: {ret[2]:4.3f}')

## TTL or External Trigger based measurements

In [ ]:
def ttl_measurement_callback(result: lt.MeasurementResultTTL):
    if result.error_code == lt.ErrorType.Success:
        print (f'{result.timestamp}\t{result.measured_point.x:4.3f}\t{result.measured_point.y:4.3f}\t{result.measured_point.z:4.3f}')
    else:
        print (f'{result.timestamp}\tError in measurement')

In [ ]:
ret = dev.start_ttl_measurement(ttl_measurement_callback)
if ret != lt.ErrorType.Success:
    msg = dev.get_error_message(ret)
    print (f'Start TTL Measurement Failed: {msg}')

In [ ]:
ret = dev.stop_ttl_measurement()
if ret != lt.ErrorType.Success:
    msg = dev.get_error_message(ret)
    print (f'Stop TTL Measurement Failed: {msg}')

## Disconnection and deletion of the instance

In [ ]:
dev.disconnect()

In [ ]:
del dev